# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/28-PythonMetinSiniflandirma.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 28 - Python ile Metin Sınıflandırma: Spam Tespiti ve Duygu Analizi

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Bir önceki derste metinleri sayısal verilere dönüştürmeyi öğrendik.

Bu derste artık bu sayısal temsilleri gerçek bir makine öğrenmesi probleminde kullanacağız.

İki uygulama geliştireceğiz:

1. **Spam / Normal Mesaj Sınıflandırma**
2. **Olumlu / Olumsuz Duygu Analizi**

Bu dersin sonunda öğrencinin:

- etiketli metin veri kümesi hazırlayabilmesi,
- metin ve hedef sütunlarını ayırabilmesi,
- stratified train-test ayrımı yapabilmesi,
- TF-IDF + sınıflandırma Pipeline oluşturabilmesi,
- Multinomial Naive Bayes kullanabilmesi,
- Logistic Regression kullanabilmesi,
- Linear SVM kullanabilmesi,
- accuracy, precision, recall ve F1-score hesaplayabilmesi,
- confusion matrix oluşturabilmesi,
- sınıflandırma hatalarını inceleyebilmesi,
- yeni metin için sınıf tahmini yapabilmesi,
- tahmin olasılıklarını yorumlayabilmesi,
- model karşılaştırabilmesi,
- duygu analizi uygulaması oluşturabilmesi,
- eğitilmiş NLP Pipeline'ını kaydedip yeniden yükleyebilmesi

hedeflenmektedir.


# 1. Metin Sınıflandırma Nedir?

Bir metni önceden belirlenmiş kategorilerden birine atama problemine **metin sınıflandırma** denir.

Örnekler:

```text
Mesaj → Spam / Normal
Yorum → Olumlu / Olumsuz
Haber → Spor / Bilim / Ekonomi
Belge → Proje / Rapor / Duyuru
E-posta → Önemli / Normal
```

Bu derste ilk olarak iki sınıflı problemler üzerinde çalışacağız.


# 2. Metin Sınıflandırma Akışı

```text
Etiketli Metinler
↓
Train / Test
↓
TF-IDF
↓
Sınıflandırma Modeli
↓
fit()
↓
predict()
↓
Accuracy / Precision / Recall / F1
↓
Yeni Metin Tahmini
```

Buradaki en önemli nokta, metin dönüştürme ve model adımlarını tek Pipeline içinde yönetmektir.


# 3. Neden Pipeline Kullanıyoruz?

Metin modelimiz iki parçadan oluşur:

```text
Ham Metin
↓
TfidfVectorizer
↓
Sayısal Özellikler
↓
Sınıflandırıcı
```

Pipeline sayesinde bu iki aşama tek model gibi çalışır.

Böylece yeni bir metin için doğrudan:

```python
model.predict(["yeni mesaj"])
```

kullanabiliriz.


# 4. İlk Proje: Spam / Normal Mesaj Sınıflandırma

Dersin internet bağlantısına ihtiyaç duymadan çalışabilmesi için kontrollü ve yapay Türkçe mesajlardan oluşan bir veri kümesi üreteceğiz.

Bu veri gerçek kullanıcı mesajlarını içermez.


# 5. Spam Mesaj Kalıpları

Spam örnekleri çoğunlukla:

- ödül vaadi,
- acele ettirme,
- kampanya,
- tıklama çağrısı,
- ücretsiz kazanç,
- hesap doğrulama bahanesi

gibi kalıplar içerebilir.

Ancak gerçek dünyada spam tespiti bundan çok daha karmaşıktır.


# 6. Normal Mesaj Kalıpları

Normal mesaj örnekleri:

- toplantı bilgisi,
- ders hatırlatma,
- proje durumu,
- aile mesajı,
- günlük plan,
- teknik bilgi

gibi sıradan iletişimlerden oluşabilir.


# 7. Gerekli Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split


# 8. Spam Veri Kümesini Üretmek

Aşağıdaki yapı yalnızca eğitim amacıyla Türkçe mesaj varyasyonları üretir.


In [ ]:
rng = np.random.default_rng(42)

spam_baslangic = [
    "Tebrikler",
    "Kazandınız",
    "Son fırsat",
    "Hemen işlem yapın",
    "Özel kampanya",
    "Bedava fırsat",
    "Hesabınızı doğrulayın",
    "Sürpriz ödül"
]

spam_icerik = [
    "hediyenizi almak için bağlantıya tıklayın",
    "ödülünüzü almak için hemen başvurun",
    "ücretsiz kupon için bu bağlantıyı açın",
    "hesabınızı korumak için bilgilerinizi güncelleyin",
    "bugüne özel indirim kodunu şimdi kullanın",
    "kazancınızı almak için formu doldurun",
    "sınırlı süreli fırsatı kaçırmayın",
    "ücretsiz üyelik için hemen kayıt olun"
]

normal_baslangic = [
    "Merhaba",
    "Hatırlatma",
    "Bilgi",
    "Selam",
    "Duyuru",
    "Günaydın",
    "İyi akşamlar",
    "Proje bilgisi"
]

normal_icerik = [
    "yarın ders saat dokuzda başlayacak",
    "toplantı cuma günü yapılacak",
    "proje dosyasını bugün yükledim",
    "ödev teslim tarihi pazartesi günü",
    "akşam eve gelirken ekmek alabilir misin",
    "bilgisayar laboratuvarı bugün açık",
    "sunum dosyasını ortak klasöre ekledim",
    "haftalık çalışma planı güncellendi"
]

spam_mesajlar = []
normal_mesajlar = []

for _ in range(180):
    spam_mesajlar.append(
        f"{rng.choice(spam_baslangic)}! {rng.choice(spam_icerik)}."
    )

    normal_mesajlar.append(
        f"{rng.choice(normal_baslangic)}. {rng.choice(normal_icerik)}."
    )

spam_df = pd.DataFrame({
    "Metin": spam_mesajlar + normal_mesajlar,
    "Etiket": (
        ["spam"] * len(spam_mesajlar)
        +
        ["normal"] * len(normal_mesajlar)
    )
})

spam_df = spam_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

spam_df.head()


# 9. Veri Kümesinin Boyutu

In [ ]:
print(
    spam_df.shape
)


# 10. Etiket Dağılımı

In [ ]:
etiket_sayilari = (
    spam_df["Etiket"]
    .value_counts()
)

etiket_sayilari


# 11. Etiket Dağılım Grafiği

In [ ]:
plt.bar(
    etiket_sayilari.index,
    etiket_sayilari.values
)

plt.ylabel("Mesaj Sayısı")
plt.title("Spam Veri Kümesi Etiket Dağılımı")
plt.show()


# 12. Eksik Veri Kontrolü

In [ ]:
print(
    spam_df.isna().sum()
)


# 13. Tekrar Eden Mesaj Sayısı

Kalıp tabanlı üretim yaptığımız için bazı mesajlar tekrar edebilir.

Bunu özellikle kontrol edelim.


In [ ]:
print(
    "Tekrar eden satır:",
    spam_df.duplicated().sum()
)


# 14. Tekrar Eden Mesajları Temizlemek

Modelin aynı mesajı hem train hem test tarafında görmesini azaltmak için tekrar eden satırları kaldıralım.


In [ ]:
spam_df = (
    spam_df
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    spam_df.shape
)


# 15. Temizleme Sonrası Sınıf Dağılımı

In [ ]:
print(
    spam_df["Etiket"]
    .value_counts()
)


# 16. Metin Uzunluğu Analizi

In [ ]:
spam_df["KarakterSayisi"] = (
    spam_df["Metin"]
    .str.len()
)

spam_df.groupby(
    "Etiket"
)["KarakterSayisi"].mean()


Metin uzunluğu bazı veri kümelerinde ayırt edici olabilir. Ancak bu projede temel olarak TF-IDF özelliklerini kullanacağız.


# 17. Kelime Sayısı Analizi

In [ ]:
spam_df["KelimeSayisi"] = (
    spam_df["Metin"]
    .str.split()
    .str.len()
)

spam_df.groupby(
    "Etiket"
)["KelimeSayisi"].mean()


# 18. Özellik ve Hedef

Metin sınıflandırmada:

```text
X → Metin
y → Etiket
```

olacaktır.


In [ ]:
X = spam_df["Metin"]
y = spam_df["Etiket"]

print(
    X.head()
)

print()

print(
    y.head()
)


# 19. Train-Test Ayrımı

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(
    "Train:",
    len(X_train)
)

print(
    "Test:",
    len(X_test)
)


# 20. Train-Test Sınıf Oranları

In [ ]:
print(
    y_train.value_counts(
        normalize=True
    )
)

print()

print(
    y_test.value_counts(
        normalize=True
    )
)


# 21. Baseline Model

Önce en sık görülen sınıfı tahmin eden basit bir model oluşturalım.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

baseline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer()
    ),
    (
        "model",
        DummyClassifier(
            strategy="most_frequent"
        )
    )
])

baseline.fit(
    X_train,
    y_train
)

baseline_pred = baseline.predict(
    X_test
)

print(
    baseline.score(
        X_test,
        y_test
    )
)


# 22. Model 1: Multinomial Naive Bayes

Multinomial Naive Bayes metin sınıflandırmada klasik ve hızlı yöntemlerden biridir.

Kelime sayıları veya TF-IDF gibi negatif olmayan metin özellikleriyle kullanılabilir.


In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(
                1,
                2
            )
        )
    ),
    (
        "model",
        MultinomialNB()
    )
])

nb_model.fit(
    X_train,
    y_train
)

nb_pred = nb_model.predict(
    X_test
)

print(
    "Model eğitildi."
)


# 23. Model 2: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lojistik_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(
                1,
                2
            )
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1500,
            random_state=42
        )
    )
])

lojistik_model.fit(
    X_train,
    y_train
)

lojistik_pred = lojistik_model.predict(
    X_test
)


# 24. Model 3: Linear SVM

Metin sınıflandırmada yüksek boyutlu seyrek özelliklerle güçlü sonuç verebilen modellerden biri Linear SVM'dir.


In [ ]:
from sklearn.svm import LinearSVC

svm_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(
                1,
                2
            )
        )
    ),
    (
        "model",
        LinearSVC(
            random_state=42
        )
    )
])

svm_model.fit(
    X_train,
    y_train
)

svm_pred = svm_model.predict(
    X_test
)


# 25. Değerlendirme Metrikleri

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# 26. Pozitif Sınıfı Belirlemek

Spam tespitinde özellikle:

```text
spam
```

sınıfını pozitif sınıf olarak ele alabiliriz.

Böylece:

- precision → spam dediğimiz mesajların ne kadarı gerçekten spam,
- recall → gerçek spam mesajların ne kadarını yakaladık

sorularına cevap verebiliriz.


# 27. Değerlendirme Fonksiyonu

In [ ]:
def ikili_metrikler(
    model_adi,
    y_gercek,
    y_tahmin
):
    return {
        "Model": model_adi,
        "Accuracy": accuracy_score(
            y_gercek,
            y_tahmin
        ),
        "PrecisionSpam": precision_score(
            y_gercek,
            y_tahmin,
            pos_label="spam"
        ),
        "RecallSpam": recall_score(
            y_gercek,
            y_tahmin,
            pos_label="spam"
        ),
        "F1Spam": f1_score(
            y_gercek,
            y_tahmin,
            pos_label="spam"
        )
    }


# 28. Modelleri Karşılaştırmak

In [ ]:
sonuclar = pd.DataFrame([
    ikili_metrikler(
        "Baseline",
        y_test,
        baseline_pred
    ),
    ikili_metrikler(
        "MultinomialNB",
        y_test,
        nb_pred
    ),
    ikili_metrikler(
        "Logistic Regression",
        y_test,
        lojistik_pred
    ),
    ikili_metrikler(
        "Linear SVM",
        y_test,
        svm_pred
    )
])

sonuclar


# 29. F1-Score'a Göre Sıralama

In [ ]:
sonuclar.sort_values(
    "F1Spam",
    ascending=False
)


# 30. Model Karşılaştırma Grafiği

In [ ]:
plt.bar(
    sonuclar["Model"],
    sonuclar["F1Spam"]
)

plt.ylim(0, 1)
plt.ylabel("Spam F1")
plt.title("Spam Sınıflandırma Model Karşılaştırması")
plt.xticks(rotation=30)
plt.show()


# 31. Logistic Regression Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        lojistik_pred
    )
)


# 32. Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    lojistik_pred,
    labels=[
        "normal",
        "spam"
    ]
)

print(cm)


# 33. Confusion Matrix Görselleştirmek

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "normal",
        "spam"
    ]
).plot()

plt.title(
    "Spam Sınıflandırma Confusion Matrix"
)

plt.show()


# 34. False Positive ve False Negative

Spam problemi için:

### False Positive

Normal mesajı spam olarak işaretlemek.

### False Negative

Spam mesajı normal olarak geçirmek.

Gerçek uygulamada hangisinin daha maliyetli olduğu kullanım amacına göre değişebilir.


# 35. Yanlış Tahminleri Bulmak

In [ ]:
hata_df = pd.DataFrame({
    "Metin": X_test.to_numpy(),
    "Gercek": y_test.to_numpy(),
    "Tahmin": lojistik_pred
})

hata_df = hata_df[
    hata_df["Gercek"]
    != hata_df["Tahmin"]
]

hata_df.head(10)


# 36. Hata Sayısı

In [ ]:
print(
    "Yanlış tahmin:",
    len(hata_df)
)


# 37. Yeni Mesaj Tahmini

In [ ]:
yeni_mesajlar = [
    "Tebrikler ödülünüz hazır hemen bağlantıya tıklayın",
    "Yarın saat 10'da proje toplantısı yapılacak",
    "Ücretsiz kupon için hemen kayıt olun",
    "Sunum dosyasını ortak klasöre yükledim"
]

for mesaj in yeni_mesajlar:
    tahmin = lojistik_model.predict(
        [mesaj]
    )[0]

    print(
        mesaj,
        "->",
        tahmin
    )


# 38. Logistic Regression Tahmin Olasılıkları

In [ ]:
olasiliklar = lojistik_model.predict_proba(
    yeni_mesajlar
)

siniflar = lojistik_model.named_steps[
    "model"
].classes_

print(
    siniflar
)

print(
    olasiliklar
)


# 39. Olasılıkları Okunabilir Hale Getirmek

In [ ]:
for mesaj, olasilik in zip(
    yeni_mesajlar,
    olasiliklar
):
    print()
    print(mesaj)

    for sinif, deger in zip(
        siniflar,
        olasilik
    ):
        print(
            sinif,
            "->",
            round(
                float(deger),
                3
            )
        )


Tahmin olasılığını mutlak gerçek veya garanti olarak yorumlamamalıyız.

Model yalnızca öğrendiği veri ve örüntülere göre skor üretir.


# 40. Spam Tahmin Fonksiyonu

In [ ]:
def spam_tahmin(
    model,
    mesaj
):
    sinif = model.predict(
        [mesaj]
    )[0]

    sonuc = {
        "mesaj": mesaj,
        "tahmin": sinif
    }

    if hasattr(
        model.named_steps[
            "model"
        ],
        "predict_proba"
    ):
        olasilik = model.predict_proba(
            [mesaj]
        )[0]

        classes = model.named_steps[
            "model"
        ].classes_

        sonuc[
            "olasiliklar"
        ] = {
            sinif_adi:
                round(
                    float(deger),
                    3
                )
            for sinif_adi, deger
            in zip(
                classes,
                olasilik
            )
        }

    return sonuc


In [ ]:
print(
    spam_tahmin(
        lojistik_model,
        "Hemen kayıt ol ücretsiz ödülünü kazan"
    )
)


# 41. Modelin Öğrendiği Kelimeleri İncelemek

Logistic Regression katsayıları sayesinde hangi özelliklerin hangi sınıfa doğru güçlü etki yaptığını inceleyebiliriz.


In [ ]:
tfidf = lojistik_model.named_steps[
    "tfidf"
]

siniflandirici = lojistik_model.named_steps[
    "model"
]

kelimeler = tfidf.get_feature_names_out()

print(
    "Özellik sayısı:",
    len(kelimeler)
)


# 42. Sınıf Sırası

In [ ]:
print(
    siniflandirici.classes_
)


İkili Logistic Regression'da katsayıların pozitif veya negatif yönü sınıf sırasına göre yorumlanmalıdır.


# 43. En Güçlü Özellikleri Bulmak

In [ ]:
katsayi = (
    siniflandirici.coef_[0]
)

en_negatif = np.argsort(
    katsayi
)[:15]

en_pozitif = np.argsort(
    katsayi
)[-15:][::-1]

print(
    "Negatif yöndeki özellikler:"
)

for i in en_negatif:
    print(
        kelimeler[i],
        round(
            float(
                katsayi[i]
            ),
            3
        )
    )

print()

print(
    "Pozitif yöndeki özellikler:"
)

for i in en_pozitif:
    print(
        kelimeler[i],
        round(
            float(
                katsayi[i]
            ),
            3
        )
    )


Bu liste modelin hangi kelime ve bigramlardan yararlandığını anlamamıza yardımcı olur.

Ancak katsayıları nedensellik olarak yorumlamamalıyız.


# 44. Unigram ve Bigram Karşılaştırması

Aynı modeli yalnızca unigram ile eğitelim.


In [ ]:
unigram_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(
                1,
                1
            )
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1500,
            random_state=42
        )
    )
])

unigram_model.fit(
    X_train,
    y_train
)

unigram_pred = unigram_model.predict(
    X_test
)

print(
    "Unigram F1:",
    f1_score(
        y_test,
        unigram_pred,
        pos_label="spam"
    )
)

print(
    "Unigram + Bigram F1:",
    f1_score(
        y_test,
        lojistik_pred,
        pos_label="spam"
    )
)


Tek bir veri kümesinde daha yüksek skor elde etmek, ilgili n-gram ayarının her problemde daha iyi olacağı anlamına gelmez.


# 45. Character N-Gram Kavramı

Metin sınıflandırmada bazen kelime n-gramları yerine karakter n-gramları da kullanılabilir.

Örneğin:

```text
kazandınız
```

kelimesinden:

```text
kaz
aza
zan
and
...
```

gibi karakter parçaları çıkarılabilir.

Bu yaklaşım yazım varyasyonlarına karşı yararlı olabilir.


# 46. Character N-Gram Modeli

In [ ]:
char_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(
                3,
                5
            ),
            min_df=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1500,
            random_state=42
        )
    )
])

char_model.fit(
    X_train,
    y_train
)

char_pred = char_model.predict(
    X_test
)

print(
    "Character N-Gram F1:",
    f1_score(
        y_test,
        char_pred,
        pos_label="spam"
    )
)


# 47. Word ve Character Modellerini Karşılaştırmak

In [ ]:
metin_model_karsilastirma = pd.DataFrame([
    {
        "Model":
            "Word TF-IDF",
        "F1":
            f1_score(
                y_test,
                lojistik_pred,
                pos_label="spam"
            )
    },
    {
        "Model":
            "Character TF-IDF",
        "F1":
            f1_score(
                y_test,
                char_pred,
                pos_label="spam"
            )
    }
])

metin_model_karsilastirma


# 48. Cross Validation ile Daha Sağlam Karşılaştırma

Tek train-test ayrımı yerine geliştirme verisi üzerinde Stratified Cross Validation kullanabiliriz.


In [ ]:
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_f1 = cross_val_score(
    lojistik_model,
    X,
    y,
    cv=cv,
    scoring="f1_macro"
)

print(
    cv_f1
)

print(
    "Ortalama:",
    cv_f1.mean()
)


Bu küçük ve yapay veri kümesinde sonuçlar çok yüksek olabilir.

Gerçek dünyadaki spam mesajları daha çeşitli olduğu için gerçek performans farklı olacaktır.


# 49. GridSearchCV ile TF-IDF Ayarlarını Aramak

Şimdi hem vectorizer hem Logistic Regression hiperparametrelerini birlikte deneyelim.


In [ ]:
from sklearn.model_selection import GridSearchCV

arama_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1500,
            random_state=42
        )
    )
])

param_grid = {
    "tfidf__ngram_range": [
        (1, 1),
        (1, 2)
    ],
    "tfidf__min_df": [
        1,
        2
    ],
    "model__C": [
        0.5,
        1.0,
        2.0
    ]
}

grid = GridSearchCV(
    arama_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)

grid.fit(
    X,
    y
)

print(
    grid.best_params_
)

print(
    grid.best_score_
)


# 50. Spam Pipeline'ını Kaydetmek

In [ ]:
import joblib

joblib.dump(
    lojistik_model,
    "28-spam-siniflandirma-modeli.joblib"
)

print(
    "Spam modeli kaydedildi."
)


# 51. Spam Modelini Yeniden Yüklemek

In [ ]:
spam_model_yuklu = joblib.load(
    "28-spam-siniflandirma-modeli.joblib"
)

print(
    spam_model_yuklu.predict([
        "Yarın proje toplantımız saat 14.00'te başlayacak"
    ])
)


# 52. İkinci Proje: Duygu Analizi

Şimdi benzer NLP zincirini:

```text
Olumlu / Olumsuz
```

duygu sınıflandırması için kullanacağız.


# 53. Duygu Analizi Nedir?

Bir metinde ifade edilen görüşün veya duygusal yönelimin belirlenmesine **duygu analizi** denir.

Basit iki sınıflı örnek:

```text
"Bu ürünü çok sevdim" → olumlu
"Hiç memnun kalmadım" → olumsuz
```

Gerçek duygu analizi sistemleri:

- nötr,
- karma,
- farklı duygu türleri

gibi daha ayrıntılı sınıflar da içerebilir.


# 54. Duygu Veri Kümesini Oluşturmak

Yine internet bağlantısına ihtiyaç duymayan, tamamen yapay Türkçe örnekler kullanacağız.


In [ ]:
olumlu_baslangic = [
    "Gerçekten",
    "Bence",
    "Bu çalışma",
    "Bu uygulama",
    "Bu deneyim",
    "Sonuç",
    "Proje",
    "Ders"
]

olumlu_icerik = [
    "çok başarılı oldu",
    "beklediğimden daha iyi çıktı",
    "çok kullanışlı ve anlaşılır",
    "beni oldukça memnun etti",
    "harika sonuç verdi",
    "çok güzel hazırlanmış",
    "öğrenmeyi kolaylaştırdı",
    "çok faydalı oldu"
]

olumsuz_baslangic = [
    "Maalesef",
    "Bence",
    "Bu çalışma",
    "Bu uygulama",
    "Bu deneyim",
    "Sonuç",
    "Proje",
    "Ders"
]

olumsuz_icerik = [
    "hiç iyi olmadı",
    "beklediğim kadar başarılı değildi",
    "çok karışık ve kullanışsız",
    "beni memnun etmedi",
    "kötü sonuç verdi",
    "yeterince anlaşılır değildi",
    "öğrenmeyi zorlaştırdı",
    "pek faydalı olmadı"
]

olumlu_metinler = []
olumsuz_metinler = []

for _ in range(180):
    olumlu_metinler.append(
        f"{rng.choice(olumlu_baslangic)} {rng.choice(olumlu_icerik)}."
    )

    olumsuz_metinler.append(
        f"{rng.choice(olumsuz_baslangic)} {rng.choice(olumsuz_icerik)}."
    )

duygu_df = pd.DataFrame({
    "Metin":
        olumlu_metinler
        +
        olumsuz_metinler,
    "Etiket":
        ["olumlu"] * len(
            olumlu_metinler
        )
        +
        ["olumsuz"] * len(
            olumsuz_metinler
        )
})

duygu_df = (
    duygu_df
    .drop_duplicates()
    .sample(
        frac=1,
        random_state=42
    )
    .reset_index(
        drop=True
    )
)

duygu_df.head()


# 55. Duygu Veri Kümesi Boyutu

In [ ]:
print(
    duygu_df.shape
)


# 56. Duygu Etiket Dağılımı

In [ ]:
print(
    duygu_df[
        "Etiket"
    ].value_counts()
)


# 57. Train-Test Ayrımı

In [ ]:
X_duygu = (
    duygu_df["Metin"]
)

y_duygu = (
    duygu_df["Etiket"]
)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_duygu,
    y_duygu,
    test_size=0.25,
    random_state=42,
    stratify=y_duygu
)


# 58. Duygu Analizi Pipeline

Olumsuzluk ifadelerini daha iyi yakalayabilmek için unigram + bigram kullanacağız.


In [ ]:
duygu_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(
                1,
                2
            )
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1500,
            random_state=42
        )
    )
])

duygu_model.fit(
    X_train_d,
    y_train_d
)

duygu_pred = duygu_model.predict(
    X_test_d
)


# 59. Duygu Modeli Accuracy

In [ ]:
print(
    "Accuracy:",
    accuracy_score(
        y_test_d,
        duygu_pred
    )
)


# 60. Duygu Classification Report

In [ ]:
print(
    classification_report(
        y_test_d,
        duygu_pred
    )
)


# 61. Duygu Confusion Matrix

In [ ]:
duygu_cm = confusion_matrix(
    y_test_d,
    duygu_pred,
    labels=[
        "olumsuz",
        "olumlu"
    ]
)

ConfusionMatrixDisplay(
    confusion_matrix=duygu_cm,
    display_labels=[
        "olumsuz",
        "olumlu"
    ]
).plot()

plt.title(
    "Duygu Analizi Confusion Matrix"
)

plt.show()


# 62. Yeni Yorumları Tahmin Etmek

In [ ]:
yeni_yorumlar = [
    "Bu proje gerçekten çok güzel oldu",
    "Hiç memnun kalmadım ve sonuç kötü",
    "Uygulama oldukça kullanışlı",
    "Bu çalışma beklediğim kadar iyi değildi"
]

for yorum in yeni_yorumlar:
    tahmin = duygu_model.predict(
        [yorum]
    )[0]

    print(
        yorum,
        "->",
        tahmin
    )


# 63. Duygu Tahmin Olasılıkları

In [ ]:
duygu_olasilik = (
    duygu_model.predict_proba(
        yeni_yorumlar
    )
)

duygu_siniflari = (
    duygu_model.named_steps[
        "model"
    ].classes_
)

for yorum, olasilik in zip(
    yeni_yorumlar,
    duygu_olasilik
):
    print()
    print(yorum)

    for sinif, deger in zip(
        duygu_siniflari,
        olasilik
    ):
        print(
            sinif,
            round(
                float(deger),
                3
            )
        )


# 64. Olumsuzluk Kelimeleri Neden Önemlidir?

Şu iki cümleyi karşılaştıralım:

```text
çok iyi
çok iyi değil
```

Tek başına `iyi` kelimesine bakmak ikinci cümleyi yanlış yorumlamaya neden olabilir.

Bigram:

```text
iyi değil
```

ifadesini ayrı özellik olarak yakalayabilir.

Bu nedenle n-gramlar duygu analizinde yararlı olabilir.


# 65. Duygu Modelinin Öğrendiği Özellikleri İncelemek

In [ ]:
duygu_tfidf = (
    duygu_model.named_steps[
        "tfidf"
    ]
)

duygu_lr = (
    duygu_model.named_steps[
        "model"
    ]
)

duygu_kelimeler = (
    duygu_tfidf
    .get_feature_names_out()
)

duygu_katsayi = (
    duygu_lr.coef_[0]
)

neg_idx = np.argsort(
    duygu_katsayi
)[:15]

poz_idx = np.argsort(
    duygu_katsayi
)[-15:][::-1]

print(
    "Bir yöne güçlü özellikler:"
)

for i in neg_idx:
    print(
        duygu_kelimeler[i],
        round(
            float(
                duygu_katsayi[i]
            ),
            3
        )
    )

print()

print(
    "Diğer yöne güçlü özellikler:"
)

for i in poz_idx:
    print(
        duygu_kelimeler[i],
        round(
            float(
                duygu_katsayi[i]
            ),
            3
        )
    )


Katsayı yönünü doğru yorumlamak için model sınıf sırasını kontrol edelim.


In [ ]:
print(
    duygu_lr.classes_
)


# 66. Duygu Analizi Hata İncelemesi

In [ ]:
duygu_hata = pd.DataFrame({
    "Metin":
        X_test_d.to_numpy(),
    "Gercek":
        y_test_d.to_numpy(),
    "Tahmin":
        duygu_pred
})

duygu_hata = duygu_hata[
    duygu_hata["Gercek"]
    != duygu_hata["Tahmin"]
]

duygu_hata.head(10)


# 67. Duygu Modelini Kaydetmek

In [ ]:
joblib.dump(
    duygu_model,
    "28-duygu-analizi-modeli.joblib"
)

print(
    "Duygu modeli kaydedildi."
)


# 68. Duygu Modelini Yüklemek

In [ ]:
duygu_model_yuklu = joblib.load(
    "28-duygu-analizi-modeli.joblib"
)

print(
    duygu_model_yuklu.predict([
        "Bu ders oldukça faydalı ve anlaşılırdı"
    ])
)


# 69. Duygu Tahmin Fonksiyonu

In [ ]:
def duygu_tahmin(
    model,
    metin
):
    tahmin = model.predict(
        [metin]
    )[0]

    sonuc = {
        "metin": metin,
        "duygu": tahmin
    }

    if hasattr(
        model.named_steps[
            "model"
        ],
        "predict_proba"
    ):
        olasilik = (
            model.predict_proba(
                [metin]
            )[0]
        )

        siniflar = (
            model.named_steps[
                "model"
            ].classes_
        )

        sonuc[
            "olasiliklar"
        ] = {
            sinif_adi:
                round(
                    float(deger),
                    3
                )
            for sinif_adi, deger
            in zip(
                siniflar,
                olasilik
            )
        }

    return sonuc


In [ ]:
print(
    duygu_tahmin(
        duygu_model_yuklu,
        "Bu çalışma gerçekten çok güzel hazırlanmış"
    )
)


# 70. Spam ve Duygu Analizi Arasındaki Ortak Yapı

İki uygulamada da aynı makine öğrenmesi iskeletini kullandık:

```text
Ham Metin
↓
TfidfVectorizer
↓
Logistic Regression
↓
Tahmin
```

Değişen şey:

- veri,
- hedef etiket,
- problem amacı.

Bu durum Pipeline bilgisinin farklı NLP problemlerine tekrar uygulanabildiğini gösterir.


# 71. Metin Sınıflandırmada Veri Kalitesi

Gerçek projelerde şu problemler görülebilir:

- hatalı etiket,
- tekrar eden veri,
- çok kısa mesajlar,
- farklı diller,
- yazım hataları,
- URL'ler,
- kullanıcı adları,
- emoji,
- spam mesajların sürekli biçim değiştirmesi.

Modelin başarısı büyük ölçüde veri kalitesine bağlıdır.


# 72. Veri Sızıntısı: Aynı Mesajın Train ve Test'te Olması

Aynı veya neredeyse aynı mesaj hem eğitim hem test verisinde bulunursa model başarısı olduğundan yüksek görünebilir.

Bu nedenle bu derste tekrar eden satırları train-test ayrımından önce temizledik.


# 73. Veri Sızıntısı: TF-IDF'i Önceden Fit Etmek

Yanlış yaklaşım:

```text
Bütün metinlerde TF-IDF fit
↓
Train-Test ayır
```

Bu işlem test verisinin kelime istatistiklerini eğitim ön işlemesine karıştırabilir.

Doğru yaklaşım:

```text
Train-Test ayır
↓
Pipeline.fit(train)
↓
Pipeline.predict(test)
```

şeklindedir.


# 74. Sınıf Dengesizliği

Gerçek spam sistemlerinde:

```text
Normal → %95
Spam → %5
```

gibi dengesiz dağılımlar olabilir.

Bu durumda yalnızca accuracy değerine bakmak yanıltıcı olabilir.

Özellikle:

- spam precision,
- spam recall,
- spam F1,
- confusion matrix

incelenmelidir.


# 75. Threshold Kavramına Giriş

Logistic Regression gibi modeller olasılık üretir.

Varsayılan sınıf kararı belirli bir karar eşiğine dayanır.

Bazı uygulamalarda:

```text
spam olasılığı > 0.80
```

gibi daha yüksek eşik istenebilir.

Bu işlem precision ve recall dengesini değiştirir.

Threshold optimizasyonunu daha ileri model değerlendirme çalışmalarında kullanabiliriz.


# 76. Manuel Threshold Örneği

In [ ]:
ornek_mesajlar = [
    "Hemen ödülünüzü almak için kayıt olun",
    "Yarın ders saat 9'da başlayacak"
]

proba = lojistik_model.predict_proba(
    ornek_mesajlar
)

spam_index = list(
    lojistik_model.named_steps[
        "model"
    ].classes_
).index(
    "spam"
)

for mesaj, p in zip(
    ornek_mesajlar,
    proba[:, spam_index]
):
    karar = (
        "spam"
        if p >= 0.80
        else "normal"
    )

    print(
        mesaj,
        "->",
        round(
            float(p),
            3
        ),
        "->",
        karar
    )


# 77. Gerçek Duygu Analizinin Zorlukları

Şu cümleleri düşünelim:

```text
Harika, yine uygulama çöktü.
Fena değil.
Beklediğim kadar kötü değildi.
Çok iyi görünüyor ama çalışmıyor.
```

Bunlar:

- ironi,
- olumsuzluk,
- bağlam,
- karma duygu

içerebilir.

Basit TF-IDF modelleri bu tür dil özelliklerinde zorlanabilir.


# 78. Modern NLP'ye Köprü

Klasik NLP yaklaşımımız:

```text
Metin
↓
TF-IDF
↓
Makine Öğrenmesi
```

Modern yaklaşımda:

```text
Metin
↓
Embedding / Transformer
↓
Derin Öğrenme Modeli
```

kullanılabilir.

Ancak TF-IDF tabanlı yöntemler:

- hızlı,
- açıklanabilir,
- düşük maliyetli,
- küçük veri kümelerinde etkili

oldukları için önemli bir temel oluşturmaya devam eder.


# 79. Flask ile Spam Tespit Uygulaması

Web uygulamasında:

```text
HTML Textarea
↓
Kullanıcı Mesajı
↓
POST
↓
Flask Route
↓
Model.predict()
↓
Spam / Normal
↓
Jinja
```

akışı oluşturulabilir.


# 80. Tkinter ile Duygu Analizi

Masaüstü uygulamasında:

```text
Text Kutusu
↓
Analiz Et Butonu
↓
Duygu Modeli
↓
Olumlu / Olumsuz
↓
Label
```

akışı kurulabilir.


# 81. API ile Metin Sınıflandırma

Örnek istek:

```json
{
    "text": "Tebrikler ödülünüz hazır"
}
```

cevap:

```json
{
    "label": "spam"
}
```

olabilir.

Aynı model başka uygulamalar tarafından API üzerinden kullanılabilir.


# 82. Bir NLP Modelinin Dosya Yapısı

```text
metin_uygulamasi/
│
├── model_egit.py
├── tahmin.py
├── model/
│   ├── spam_modeli.joblib
│   └── duygu_modeli.joblib
├── veri/
│   └── mesajlar.csv
└── app.py
```

Model eğitimi ile kullanıcı uygulamasını birbirinden ayırmak proje düzenini iyileştirir.


# 83. Etik ve Gizlilik

Mesaj ve yorum verileri kişisel bilgi içerebilir.

Gerçek NLP projelerinde:

- veri kullanım izni,
- anonimleştirme,
- kişisel veri koruma,
- güvenli depolama,
- model çıktısının yanlış kullanılmaması

önemlidir.

Özellikle özel mesajları izinsiz biçimde eğitim verisi olarak kullanmamalıyız.


# 84. Spam Modeli Kusursuz Değildir

Model:

```text
spam
```

tahmini verdiğinde mesaj kesin spam değildir.

Aynı şekilde:

```text
normal
```

tahmini güvenli olduğunun garantisi değildir.

Gerçek sistemlerde otomatik kararlar ek kontrollerle desteklenebilir.


# 85. Duygu Analizi İnsan Duygusunu Tam Olarak Ölçmez

Duygu modeli yalnızca yazılı metindeki örüntülere göre sınıf üretir.

Bir kişinin:

- gerçek duygusunu,
- psikolojik durumunu,
- niyetini

kesin olarak ölçmez.

Bu ayrım özellikle insanlarla ilgili NLP uygulamalarında önemlidir.


# 86. Proje Akışı

```text
Problem
↓
Etiketli Metin
↓
Veri Kalitesi
↓
Train / Test
↓
TF-IDF
↓
Baseline
↓
Naive Bayes / Logistic Regression / Linear SVM
↓
Accuracy / Precision / Recall / F1
↓
Confusion Matrix
↓
Hata Analizi
↓
N-Gram Karşılaştırması
↓
Cross Validation
↓
Grid Search
↓
Yeni Metin Tahmini
↓
Pipeline Kaydı
```


# 87. Ders Özeti

Bu derste:

- metin sınıflandırma,
- etiketli metin veri kümesi,
- spam / normal sınıflandırma,
- duygu analizi,
- TfidfVectorizer,
- Pipeline,
- DummyClassifier,
- MultinomialNB,
- Logistic Regression,
- LinearSVC,
- accuracy,
- precision,
- recall,
- F1,
- confusion matrix,
- classification report,
- hata analizi,
- `predict_proba()`,
- unigram,
- bigram,
- character n-gram,
- Cross Validation,
- GridSearchCV,
- threshold kavramı,
- model kaydetme ve yükleme,
- Flask / Tkinter / API entegrasyonu

konularını uygulamalı olarak öğrendik.


# 88. Mini Uygulamalar

1. 100 normal ve 100 spam mesajdan oluşan veri kümesi hazırlayın.
2. Tekrar eden mesajları temizleyin.
3. Etiket dağılımını gösterin.
4. Train-test ayrımı yapın.
5. DummyClassifier baseline oluşturun.
6. MultinomialNB modeli oluşturun.
7. Logistic Regression modeli oluşturun.
8. LinearSVC modeli oluşturun.
9. Accuracy değerlerini karşılaştırın.
10. Spam precision değerlerini karşılaştırın.
11. Spam recall değerlerini karşılaştırın.
12. Spam F1 değerlerini karşılaştırın.
13. Confusion matrix oluşturun.
14. Yanlış tahmin edilen mesajları listeleyin.
15. Yeni mesaj tahmin fonksiyonu yazın.
16. Logistic Regression olasılıklarını gösterin.
17. Unigram ve bigram modellerini karşılaştırın.
18. Character n-gram modeli oluşturun.
19. Cross Validation yapın.
20. GridSearchCV ile n-gram ayarı arayın.
21. Duygu analizi veri kümesi hazırlayın.
22. Olumlu / olumsuz model eğitin.
23. Duygu analizi için yeni yorum tahmini yapın.
24. İki Pipeline'ı `joblib` ile kaydedin.
25. Spam veya duygu modelini Flask/Tkinter arayüzüne bağlamak için uygulama taslağı hazırlayın.


# 89. Yapay Zeka Proje Görevi

Bir **Türkçe Metin Sınıflandırma Sistemi** geliştirin.

Konu seçenekleri:

- spam / normal,
- olumlu / olumsuz,
- proje kategorisi,
- haber kategorisi,
- teknik destek talebi kategorisi,
- konu sınıflandırma.

Projede en az:

- 300 metin,
- en az 2 sınıf,
- tekrar veri kontrolü,
- train-test ayrımı,
- baseline,
- TF-IDF,
- unigram ve bigram karşılaştırması,
- MultinomialNB,
- Logistic Regression,
- LinearSVC,
- accuracy,
- precision,
- recall,
- F1,
- confusion matrix,
- hata analizi,
- Cross Validation,
- GridSearchCV,
- yeni metin tahmini,
- Pipeline kaydı

bulunsun.

Ek geliştirme:

- Flask web arayüzü,
- Tkinter masaüstü uygulaması,
- JSON API,
- SQLite tahmin geçmişi

özelliklerinden biri eklenebilir.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin aşağıdaki gerçek NLP sınıflandırma zincirini kurabilmesi hedeflenmektedir:

**Ham Metin**

↓

**Etiketli Veri**

↓

**Train / Test**

↓

**TF-IDF**

↓

**Naive Bayes / Logistic Regression / SVM**

↓

**Tahmin**

↓

**Precision / Recall / F1**

↓

**Confusion Matrix**

↓

**Hata Analizi**

↓

**Yeni Metin**

↓

**Pipeline Kaydı**

Bu noktada öğrenciler artık metni yalnızca sayısallaştırmakla kalmıyor; Python ile metni otomatik sınıflandıran çalışan yapay zeka uygulamaları geliştirebiliyor.

Bir sonraki dersimizde **görüntü işleme ve bilgisayarlı görü** alanına giriş yapacağız.
